In [2]:
import sys

# Make sure your notebook kernel is the conda env: py312pt291cu128
print("python executable:", sys.executable)

import memengine
print("memengine:", memengine)
print("memengine exports:", [n for n in ("FUMemory","STMemory","LTMemory","MBMemory","MemoryConfig") if hasattr(memengine,n)])

from memengine import MemoryConfig, FUMemory, STMemory, LTMemory, MBMemory

python executable: d:\Anaconda\envs\py312pt291cu128\python.exe
memengine: <module 'memengine' from 'd:\\Anaconda\\envs\\py312pt291cu128\\Lib\\site-packages\\memengine\\__init__.py'>
memengine exports: ['FUMemory', 'STMemory', 'LTMemory', 'MBMemory', 'MemoryConfig']


In [3]:
import json
from pathlib import Path

DATA_PATH = Path("locomo10.json")
assert DATA_PATH.exists(), f"Not found: {DATA_PATH.resolve()}"

raw = json.loads(DATA_PATH.read_text(encoding="utf-8"))
print("items:", len(raw))
print("keys of first item:", list(raw[0].keys()))

qa = raw[0]["qa"]
print("qa count:", len(qa))
print("sample qa[0] keys:", list(qa[0].keys()))

# Turn qa pairs into memory observations (plain text)
# (Some entries may miss the 'answer' field; we skip those.)
qa_valid = [x for x in qa if "answer" in x]
observations = [
    f"Q: {x['question']}\nA: {x['answer']}\nEvidence: {', '.join(x.get('evidence', []))}"
    for x in qa_valid
]

# Pick one query to test recall (use an existing question)
query = qa_valid[0]["question"]
print("query:", query)
print("observation[0]:\n", observations[0])

items: 10
keys of first item: ['qa', 'conversation', 'event_summary', 'observation', 'session_summary', 'sample_id']
qa count: 199
sample qa[0] keys: ['question', 'answer', 'evidence', 'category']
query: When did Caroline go to the LGBTQ support group?
observation[0]:
 Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3


In [ ]:
def make_common_config(*, usable_gpu: str = "", display_method: str = "ScreenDisplay") -> dict:
    # Minimal, runnable configs derived from the official open-source defaults,
    # but adapted to run locally without external model paths.
    return {
        "global_config": {"usable_gpu": usable_gpu},
        "storage": {},
        "display": {
            "method": display_method,
            "prefix": "----- Current Memory Start (%s) -----",
            "suffix": "----- Current Memory End -----",
            "key_format": "(%s)",
            "key_value_sep": "\n",
            "item_sep": "\n",
            # FileDisplay-only arg (ignored by ScreenDisplay)
            "output_path": "logs/sample.log",
        },
        "recall": {
            "truncation": {
                "method": "LMTruncation",
                "mode": "word",
                "number": 256,
                "path": "",  # only used for token-based truncation
            },
            "utilization": {
                "method": "ConcateUtilization",
                "prefix": "[Memory Start]",
                "suffix": "[Memory End]",
                "list_config": {"index": True, "sep": "\n"},
                "dict_config": {"key_format": "(%s)", "key_value_sep": "\n", "item_sep": "\n"},
            },
            "empty_memory": "None",
        },
        "store": {},
    }


def make_text_retrieval_config(*, topk: int = 5, st_model: str = "sentence-transformers/all-MiniLM-L6-v2") -> dict:
    # LTMemory / MBMemory rely on embeddings; this uses SentenceTransformers.
    return {
        "method": "TextRetrieval",
        "encoder": {
            "method": "STEncoder",
            "name": st_model,
            "dimension": 384,
            "path": st_model,
        },
        "mode": "cosine",
        "topk": topk,
    }


def run_memory(memory, obs_list, query_text, *, with_time: bool = False, fixed_time: int | None = None):
    memory.reset()
    for i, obs in enumerate(obs_list):
        if with_time:
            t = fixed_time if fixed_time is not None else i
            memory.store({"text": obs, "time": t})
        else:
            memory.store(obs)
    return memory.recall(query_text)

In [ ]:
# --- FUMemory (Full / long-context) ---
fu_cfg = make_common_config()
fu_cfg["name"] = "FUMemory"
fu_cfg["store"] = {"method": "FUMemoryStore"}
fu_cfg["recall"]["method"] = "FUMemoryRecall"

fu = FUMemory(MemoryConfig(fu_cfg))
fu_ans = run_memory(fu, observations[:30], query)
print("FUMemory recall result:\n", fu_ans)

# Optional: visualize internal storage
fu.display()

In [ ]:
# --- STMemory (Short-term / recent window) ---
st_cfg = make_common_config()
st_cfg["name"] = "STMemory"
st_cfg["store"] = {"method": "LTMemoryStore"}
st_cfg["recall"].update({
    "method": "STMemoryRecall",
    "time_retrieval": {"method": "TimeRetrieval", "mode": "raw", "topk": 5},
})

st = STMemory(MemoryConfig(st_cfg))
st_ans = run_memory(st, observations[:30], query)
print("STMemory recall result:\n", st_ans)

st.display()

In [ ]:
# --- LTMemory (Long-term / embedding retrieval) ---
lt_cfg = make_common_config()
lt_cfg["name"] = "LTMemory"
lt_cfg["store"] = {"method": "LTMemoryStore"}
lt_cfg["recall"].update({
    "method": "LTMemoryRecall",
    "text_retrieval": make_text_retrieval_config(topk=5),
})

lt = LTMemory(MemoryConfig(lt_cfg))
lt_ans = run_memory(lt, observations[:80], query)
print("LTMemory recall result:\n", lt_ans)

lt.display()

In [ ]:
# --- MBMemory (MemoryBank) ---
# Note: MBMemory's default store op may call an LLM summarizer when the "day" (time) changes.
# To keep this demo fully local/offline, we store all observations with the SAME time value,
# so summarization is never triggered, while recall (embedding-based) still works.

mb_cfg = make_common_config()
mb_cfg["name"] = "MBMemory"
mb_cfg["store"] = {
    "method": "MBMemoryStore",
    "summarizer": {
        "method": "LLMSummarizer",
        "LLM_config": {
            "method": "APILLM",
            "name": "gpt-4o-mini",
            "api_key": "DUMMY",
            "base_url": "https://api.openai.com/v1",
            "temperature": 0.0,
        },
        "prompt": {
            "template": "Content: {content}\nSummarize the above content concisely, extracting the main themes and key information.",
            "input_variables": ["content"],
        },
    },
}
mb_cfg["recall"].update({
    "method": "MBMemoryRecall",
    "text_retrieval": make_text_retrieval_config(topk=5),
    # omit "forget" in this basic demo to make results deterministic/visible
})

mb = MBMemory(MemoryConfig(mb_cfg))
mb_ans = run_memory(mb, observations[:80], query, with_time=True, fixed_time=0)
print("MBMemory recall result:\n", mb_ans)

mb.display()